In [ ]:
import os
os.chdir("..")

import torch

device = torch.device("cuda:0")
dtype = torch.bfloat16  # Image and video; audio uses float32 below.


# Image

## KVAE-2d

KVAE 3.0 is selected by default. The previous image models remain available in `model_paths`.


In [ ]:
from kvae.models import KVAEImage

model_paths = {
    "KVAE_1.0": "kandinskylab/KVAE-2D-1.0",
    "KVAE_2.0": "kandinskylab/KVAE-2D-2.0",
    "KVAE_3.0": "kandinskylab/KVAE-2D-3.0",
}

model = "KVAE_3.0"

vae = KVAEImage.from_pretrained(model_paths[model], strict=True).eval().to(device=device, dtype=dtype)


## Metrics

In [ ]:
from torchmetrics import MetricCollection
from torchmetrics.image import  PeakSignalNoiseRatio, LearnedPerceptualImagePatchSimilarity

metrics = MetricCollection(
    {
        "psnr": PeakSignalNoiseRatio(data_range=(-1, 1), reduction="none", dim=[1, 2, 3]),
        "lpips": LearnedPerceptualImagePatchSimilarity(reduction="none", net_type="alex"),
    }
).to(device=device)

## Prepare image

Use an RGB image with height and width divisible by 8.


In [ ]:
from pathlib import Path
from data import read_image, _norm_to_255numpy, save_tensor_image
import mediapy

image_path = Path("./assets/images/0004.png")
image = read_image(image_path).unsqueeze(0).to(dtype=dtype, device=device)

with torch.no_grad():
    latent = vae.encode(image).latent_dist.mode()
    recon = vae.decode(latent).clip(-1, 1)

    # Compute Metrics
    result = metrics(recon.float(), image.float())

print(f"PSNR (dB): {result['psnr'].item():.4f}")
print(f"LPIPS    : {result['lpips'].item():.4f}")

# Show image, reconstruction
show_image = _norm_to_255numpy(image.squeeze(0).float().cpu().permute(1,2,0), input_norm='-11')
show_recon = _norm_to_255numpy(recon.squeeze(0).float().cpu().permute(1,2,0), input_norm='-11')
show_diff = _norm_to_255numpy((abs(recon-image)-1).squeeze(0).float().cpu().permute(1,2,0), input_norm='-11')
mediapy.show_images([show_image, show_recon, show_diff], ["Input Image", "Reconstructed Image", "Difference"], height=720)

# Saving reconstruction
saving_folder = Path('./outputs/images')
save_tensor_image(
    recon.float().cpu().squeeze(0),
    save_dir_path=saving_folder,
    filename=image_path.name,
)

# Video

## KVAE-3d

KVAE 3.0 uses 4 × 32 × 32 compression with 128 latent channels. The previous video models remain available in `model_paths`.

KVAE-Video 3.0 normalizes latents per channel using fixed `latent_mean` and `latent_std` vectors stored in the checkpoint: `z_normalized = (z_raw - latent_mean) / latent_std`. The default config enables this: `encode` returns the normalized posterior, and `decode` applies the inverse transform. If you override `use_latent_norm`, use the same setting in both calls; `None` follows the config.


In [ ]:
from kvae.models import KVAEVideo

model_paths = {
    "KVAE_1.0": "kandinskylab/KVAE-3D-1.0",
    "KVAE_2.0-t4s8": "kandinskylab/KVAE-3D-2.0-t4s8",
    "KVAE_2.0-t4s16": "kandinskylab/KVAE-3D-2.0-t4s16",
    "KVAE_3.0": "kandinskylab/KVAE-3D-3.0",
}

model = "KVAE_3.0"

vae = KVAEVideo.from_pretrained(model_paths[model], strict=True).eval().to(device=device, dtype=dtype)


## Metrics

In [ ]:
from torchmetrics import MetricCollection
from metrics.video_metrics import VideoPSNR, VideoLPIPS

metrics = MetricCollection(
    {
        "psnr": VideoPSNR(data_range=(-1, 1), metric_chank_size=10),
        "lpips": VideoLPIPS(net_type="alex", metric_chank_size=10),
    }
).to(device=device)

## Prepare video

`video_path` can point to a video file or a directory of ordered PNG frames. Adjust `video_fps` for your input. Cropping and input normalization follow the model's data config; models without it use the previous defaults.

Do not process independent videos concurrently or interleave their chunks on the same `KVAEVideo` instance. Use a separate model instance for each concurrent stream.


In [ ]:
from pathlib import Path
from data import VideoReader, _norm_to_255numpy, quant_renormalization
from data.video_dataset import TruncVideo
import mediapy

video_path = Path("./assets/test1/panda_004_0.mp4")
video_fps = 29.97
seg_len = 16

data_config = getattr(vae, "data_config", {})
input_norm = data_config.get("input_norm", "m11")
reader = VideoReader(
    stream_pattern="*.png",
    input_norm=input_norm,
    transform=TruncVideo(
        spatial_compression=data_config.get("spatial_compression", 16),
        temporal_compression=data_config.get("temporal_compression", 8),
    ),
)
video = reader.read_video(video_path)["frames"].unsqueeze(0).to(dtype=dtype, device=device)

with torch.no_grad():
    # KVAE 3.0 returns per-channel normalized latents by default.
    latent = vae.encode(video, seg_len=seg_len).latent_dist.mode()
    # decode automatically restores the original per-channel latent scale.
    recon = vae.decode(latent, seg_len=seg_len).clip(-1, 1)

    # Compute Metrics
    video = quant_renormalization(video.float(), input_norm=input_norm, output_norm="-11").squeeze(0)
    recon = quant_renormalization(recon.float(), input_norm=input_norm, output_norm="-11").squeeze(0)
    result = metrics(recon, video)

print(f"PSNR (dB): {result['psnr_metric_per_video'].item():.4f}")
print(f"LPIPS    : {result['lpips_metric_per_video'].item():.4f}")

# Show video, reconstruction
show_video = _norm_to_255numpy(video.cpu().permute(0, 2, 3, 1), input_norm="-11")
show_recon = _norm_to_255numpy(recon.cpu().permute(0, 2, 3, 1), input_norm="-11")
mediapy.show_videos([show_video, show_recon], ["Input Video", "Reconstructed Video"], fps=video_fps)

# Saving reconstruction
saving_folder = Path("./outputs/videos")
saving_folder.mkdir(parents=True, exist_ok=True)
output_filepath = saving_folder / (video_path.stem + ".mp4")
mediapy.write_video(output_filepath, show_recon, fps=video_fps)
print(f"Saving path for video result:\t{str(output_filepath)}")


# Audio

## KVAE-1d

KVAE-Audio has 64 latent channels. KVAE-Audio-Stereo has 128 latent channels.


In [ ]:
from kvae.models import KVAEAudio

audio_model_paths = {
    "KVAE-Audio": "kandinskylab/KVAE-Audio",
    "KVAE-Audio-Stereo": "kandinskylab/KVAE-Audio-Stereo",
}
audio_model_name = "KVAE-Audio-Stereo"  # or "KVAE-Audio"

# Keep float32 for parity with the original KVAE-Audio inference.
audio_dtype = torch.float32
vae = KVAEAudio.from_pretrained(audio_model_paths[audio_model_name], strict=True).eval().to(device=device, dtype=audio_dtype)


## Metrics

In [ ]:
from metrics.audio_metrics import (
    MelSpectrogramDistance,
    MultiScaleSTFTDistance,
    SISDRDistance,
    WaveformL1Distance,
)

audio_distances = {
    "waveform_l1": WaveformL1Distance().to(device),
    "stft": MultiScaleSTFTDistance().to(device),
    "mel": MelSpectrogramDistance().to(device),
    "sisdr": SISDRDistance().to(device),
}


## Prepare audio

Use 48 kHz audio. Channel conversion follows `inference_1d_kvae.py`: mono input is duplicated for the stereo model and its reconstruction is averaged back to mono; the mono model processes each input channel independently and restores the original channel layout. Playback preserves the saved file's channels.


In [ ]:
from pathlib import Path

from audiotools import AudioSignal
from IPython.display import Audio, display

from data import read_audio, save_audio
from scripts.inference_1d_kvae import match_model_channels

audio_path = Path("./assets/audio_test/98537484.wav")
input_signal = read_audio(audio_path).to(device)
audio = input_signal.audio_data.to(dtype=audio_dtype)
sample_rate = input_signal.sample_rate
batch, num_input_channels, length = audio.shape
model_channels = int(vae.num_channels)
# Mono model + stereo: encode each channel independently, then restore stereo.
batched_channels = model_channels == 1 and num_input_channels > 1
# Stereo model + mono: duplicate the input, then average the reconstruction.
average_channels = model_channels > 1 and num_input_channels == 1
if batched_channels:
    model_audio = audio.reshape(batch * num_input_channels, 1, length)
else:
    model_audio = match_model_channels(audio, model_channels)

with torch.no_grad():
    posterior = vae.encode(model_audio).latent_dist
    latent = posterior.mode()
    reconstruction = vae.decode(latent)
    if batched_channels:
        reconstruction = reconstruction.reshape(batch, num_input_channels, -1)
    elif average_channels:
        reconstruction = reconstruction.mean(dim=1, keepdim=True)
    reconstruction = reconstruction[..., :length]

channels = min(audio.shape[1], reconstruction.shape[1])
reference = audio[:, :channels]
reconstruction = reconstruction[:, :channels]

reference_signal = AudioSignal(reference.float(), sample_rate)
reconstruction_signal = AudioSignal(reconstruction.float(), sample_rate)
metric_values = {
    "waveform_l1": audio_distances["waveform_l1"](
        reference_signal, reconstruction_signal
    ).item(),
    "stft": audio_distances["stft"](
        reference_signal, reconstruction_signal
    ).item(),
    "mel": audio_distances["mel"](
        reference_signal, reconstruction_signal
    ).item(),
    "sisdr_db": -audio_distances["sisdr"](
        reference_signal, reconstruction_signal
    ).item(),
}

audio_saving_folder = Path("./outputs/audio")
saved_audio_path = save_audio(
    tensor=reconstruction.float(),
    reference_tensor=reference.float(),
    sample_rate=sample_rate,
    save_dir_path=audio_saving_folder,
    filename=audio_path.name,
)
saved_reconstruction = read_audio(saved_audio_path)

print("Input shape:", tuple(audio.shape))
print("Latent shape:", tuple(latent.shape))
print("Reconstruction shape:", tuple(reconstruction.shape))
print("Sample rate:", sample_rate)
print("Metrics:", metric_values)
print("Saved to:", saved_audio_path)

print("Original")
display(Audio(input_signal.audio_data[0].detach().cpu().numpy(), rate=sample_rate))
print("Saved reconstruction")
display(Audio(saved_reconstruction.audio_data[0].numpy(), rate=sample_rate))
